**Cell #01**

# RAG11 Nutrition — Stage 1.9:
# Verify All Data

Standalone integrity check between the local files (`stage1_eda_output/`) and what is actually in
`rag11_data_sources` / `rag11_chunks_parent_table` / `rag11_chunks_child_table`. Run it whenever you want to confirm a
load landed correctly (including after a partial run, or after re-tuning stage 1.1 and reloading).

**All the code lives in `reusable_code/stage1/verify.py`**; the "expected" rows are built by the very same functions
stage 1.2 uses (`stage1/common.py`), so the two can't drift apart. The whole pipeline without notebooks:
`./run_stage1_all.command`.

For every single local record, not just row counts, it checks:

- the row exists in Supabase at all (nothing silently missing)
- its `rowJSON` matches the local file exactly (nothing corrupted or stale)
- its `rowOwnerGUID` / `rowParentGUID` / `orderInList` match what the file implies
- (child rows) an embedding is present and has the right length
- nothing is left in the tables without a local file (orphans)

**Read-only** — this notebook never writes to Supabase.

In [1]:
# Cell #02
%pip install -q -r requirements.txt


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


**Cell #03**

## Setup and local files

In [2]:
# Cell #04
from reusable_code.clients import make_supabase_client
from reusable_code.stage1 import verify as s19
from reusable_code.stage1.common import get_paths, load_local_data

supabase = make_supabase_client()
local = load_local_data(get_paths())
print("\nExpected from local files:", local.counts())

[source1] 133 parent file(s), 147 child file(s)
[source2] 136 parent file(s), 1105 child file(s)
[source3] 91 parent file(s), 820 child file(s)
[source4] 17 parent file(s), 79 child file(s)
[source5] 22 parent file(s), 163 child file(s)
[source6] 22 parent file(s), 163 child file(s)
[source7] 94 parent file(s), 181 child file(s)
[source8] 97 parent file(s), 3997 child file(s)
[source9] 35 parent file(s), 730 child file(s)
[source10] 9 parent file(s), 101 child file(s)
[source11] 30 parent file(s), 712 child file(s)
[source12] 15 parent file(s), 463 child file(s)
[source13] 44 parent file(s), 1214 child file(s)
[source14] 18 parent file(s), 368 child file(s)
[source15] 42 parent file(s), 3598 child file(s)
[source16] 245 parent file(s), 8318 child file(s)
[source17] 15 parent file(s), 1125 child file(s)
[source18] 252 parent file(s), 2889 child file(s)

Expected from local files: 18 source row(s), 1317 parent row(s), 26173 child row(s)


**Cell #05**

## Fetch every row from Supabase (paginated)

`select()` is capped per request, so this pages with `.range()` until a page comes back short.

In [3]:
# Cell #06
db = s19.fetch_db(supabase)

Fetching all source rows from Supabase...
  -> 18 row(s) in rag11_data_sources
Fetching all parent rows from Supabase...
  -> 1317 row(s) in rag11_chunks_parent_table
Fetching all child (including embeddings) rows from Supabase...
  -> 26173 row(s) in rag11_chunks_child_table


**Cell #07**

## Compare — source rows, parent rows, child rows (+ embedding presence/dimension)

In [4]:
# Cell #08
report = s19.Report()
s19.compare_sources(local, db, report)
s19.compare_parents(local, db, report)
s19.compare_children(local, db, report)

Source rows -- OK: 18, missing: 0, mismatched: 0, orphaned in DB: 0
Parent rows -- OK: 1317, missing: 0, mismatched: 0, orphaned in DB: 0
Child rows -- OK: 26173, missing: 0, mismatched: 0, bad/missing embedding: 0, orphaned in DB: 0


**Cell #09**

## Row-count summary by source

Local file counts against Supabase counts per source: the fastest way to spot a stale/orphaned source after
re-tuning stage 1.1. `difference = local - db`: positive means local has rows Supabase doesn't (not yet upserted),
negative means Supabase has rows the local files don't (orphans).

In [5]:
# Cell #10
from IPython.display import HTML, display

display(HTML(s19.counts_table_html(s19.counts_table_lines(local, db))))

**Cell #11**

## Smoke-test vector search

Confirms `match_rag11_child_chunks` (the RPC from `create_sql_tables.sql`) returns results end to end, using a real
embedding already stored in the table (no Voyage call).

In [6]:
# Cell #12
s19.smoke_test(supabase, db)

Smoke-test match_rag11_child_chunks (querying with one child's own embedding):
  [source11 #6] dist=0.0000  [Source: _OceanofPDF.com_Nancy_Clarks_Sports_Nutrition_Guidebook_-_Nancy_Clark.p...
  [source11 #4] dist=0.2010  [Source: _OceanofPDF.com_Nancy_Clarks_Sports_Nutrition_Guidebook_-_Nancy_Clark.p...
  [source11 #5] dist=0.2026  [Source: _OceanofPDF.com_Nancy_Clarks_Sports_Nutrition_Guidebook_-_Nancy_Clark.p...


**Cell #13**

## Overall verdict

In [7]:
# Cell #14
s19.print_verdict(report)

PASS -- every local source/chunk file matches its row in Supabase exactly, and every child row has a valid embedding. No orphaned rows either.
